# アクリル入稿 SVG チェック

上から順に実行してください。

1. 準備
2. SVG をアップロード
3. 中身の確認（`info`）
4. **同じ形のパーツのカット線チェック**（`shapes`）
5. NG 確認用 SVG のダウンロード
6. 四隅にネジ穴を追加（`holes`）
7. 名前と名簿の照合（`names`）

判定の考え方や許容値の決め方は `tools/acrylic/README.md` を参照してください。

## 1. 準備

In [ ]:
!pip install -q svgpathtools numpy openpyxl

import os, sys, subprocess

REPO = "https://github.com/hanyowan-hue/hakumai-studio-site.git"
TOOL_DIR = "/content/hakumai-studio-site/tools/acrylic"

if not os.path.isdir(TOOL_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/hakumai-studio-site"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr)
        print("clone できませんでした。tools/acrylic フォルダを /content にアップロードして、")
        print("下の TOOL_DIR をそのパスに書き換えてください。")

sys.path.insert(0, TOOL_DIR)
os.chdir(TOOL_DIR)

from acrylic_check import geometry, namecheck, screwholes, shapecheck, svgdoc, svgedit
print("準備完了")

## 2. SVG をアップロード

In [ ]:
from google.colab import files

uploaded = files.upload()
SVG_FILE = "/content/" + list(uploaded.keys())[0]
with open(SVG_FILE, "wb") as f:
    f.write(uploaded[list(uploaded.keys())[0]])

print("対象ファイル:", SVG_FILE)

## 3. 中身の確認

カット線がどのレイヤーに入っているかを確認します。
カット線以外（印刷用の figure など）も拾ってしまう場合は、
次のセルの `LAYER` に正規表現を入れて絞り込んでください。

In [ ]:
LAYER = None          # 例: 'カット|cut'  （None なら全部）
DPI = 96.0            # 単位のない座標を px とみなすときの解像度

doc = svgdoc.load(SVG_FILE, dpi=DPI, layer_pattern=LAYER)

for w in dict.fromkeys(doc.warnings):
    print("⚠️", w)

print(f"縮尺: 1 ユーザー単位 = {doc.mm_per_unit:.6f} mm")
print(f"閉じた輪郭: {len(doc.contours)} 本")
print()
print(f"{'No':>4}  {'名前':<26} {'深さ':>3} {'長辺mm':>9} {'短辺mm':>9}")
for c in doc.contours:
    major, minor, _ = geometry.min_area_rect(c.points)
    print(f"{c.index:>4}  {c.name:<26} {c.depth:>3} {major:>9.3f} {minor:>9.3f}")

## 4. 同じ形のパーツのカット線チェック

- **倍率** … 基準パーツの何倍か（1.00000 なら拡大縮小なし）
- **ズレ最大mm** … ぴったり重ねたときの最大のズレ。はめ合いはこの値で判断
- **鏡像** … 裏返さないと重ならない（同じ穴に入らない）

In [ ]:
DEV_TOL_MM   = 0.10    # 形のズレの許容量（mm）
SCALE_TOL    = 0.002   # 拡大縮小の許容量（0.002 = 0.2%）
SHAPE_TOL    = 0.02    # 同じ形とみなす閾値（無次元）
ALLOW_MIRROR = False   # 鏡像を許容するなら True

report = shapecheck.analyze(
    doc,
    shape_tol=SHAPE_TOL,
    dev_tol_mm=DEV_TOL_MM,
    scale_tol=SCALE_TOL,
    allow_mirror=ALLOW_MIRROR,
)

for group in report.groups:
    print(f"---------- 形{group.number}（{len(group.members)} 個, 深さ {group.depth}） ----------")
    print(f"{'名前':<26} {'倍率':>9} {'ズレ最大mm':>11} {'ズレRMSmm':>10} {'鏡像':>5}  判定")
    for m in group.members:
        mark = "基準" if m.contour.index == group.reference.index else (
            "OK" if m.ok else "❌ " + " / ".join(m.issues))
        print(f"{m.contour.name:<26} {m.fit.scale:>9.5f} {m.fit.max_dev:>11.4f} "
              f"{m.fit.rms_dev:>10.4f} {'あり' if m.fit.mirrored else '':>5}  {mark}")
    print()

if report.singles:
    print(f"---------- 単独の形（{len(report.singles)} 個・比較対象なし） ----------")
    for g in report.singles:
        m = g.members[0]
        print(f"  {m.contour.name:<26} {m.major_mm:>9.3f} × {m.minor_mm:.3f} mm （深さ {m.contour.depth}）")
    print()

for w in dict.fromkeys(report.warnings):
    print("⚠️", w)

ng = report.ng_members
print(f"\n=== NG: {len(ng)} 件 ===")
for m in ng:
    print("  ❌", m.contour.name, ":", " / ".join(m.issues))
if not ng:
    print("  ✅ 同じ形のパーツはすべて同一のカット線です（拡大縮小・歪み・鏡像なし）。")

## 5. NG 位置の確認用 SVG

NG になった輪郭を**赤で上に重ねた**ファイルを作ります。
元のパスデータには一切触りません（重ねるだけ）。

In [ ]:
from pathlib import Path

if ng:
    src = Path(SVG_FILE).read_text(encoding="utf-8")
    out = SVG_FILE.rsplit(".", 1)[0] + "_NG確認.svg"
    Path(out).write_text(
        svgedit.insert_before_root_close(src, shapecheck.build_overlay(doc, ng)),
        encoding="utf-8",
    )
    print("書き出し:", out)
    files.download(out)
else:
    print("NG がないので確認用 SVG は作りません。")

## 6. 四隅にネジ穴を追加

4 つとも同じ直径・同じ角からの距離で入れ、
**書き出したファイルを読み直して実測**した結果を表示します。
✅ が出なければその出力は使わないでください。

In [ ]:
DIAMETER_MM = 4.0     # 穴の直径
INSET_MM    = 12.0    # 辺からの距離（角からの距離は対角方向）
RECT_MODE   = "outline"   # "outline"=最大面積の輪郭 / "all"=全図形の外接矩形

plan = screwholes.plan_holes(doc, diameter_mm=DIAMETER_MM, inset_mm=INSET_MM, mode=RECT_MODE)

x0, y0, x1, y1 = plan.rect_mm
print("基準矩形:", plan.source)
print(f"          {x1 - x0:.3f} × {y1 - y0:.3f} mm")
print(f"直径 {plan.diameter_mm:.3f}mm / 辺からの距離 X {plan.inset_x_mm:.3f}mm, Y {plan.inset_y_mm:.3f}mm")
for name, (cx, cy) in zip(["左上", "右上", "右下", "左下"], plan.centers_mm):
    print(f"  {name}: ({cx:.3f}, {cy:.3f}) mm")

out = SVG_FILE.rsplit(".", 1)[0] + "_ネジ穴.svg"
Path(out).write_text(
    screwholes.apply_to_file(Path(SVG_FILE).read_text(encoding="utf-8"), doc, plan),
    encoding="utf-8",
)
print("\n書き出し:", out)

print("\n--- 読み直して実測 ---")
for line in screwholes.verify(svgdoc.load(out, dpi=DPI), rect_mm=plan.rect_mm):
    print(" ", line)

files.download(out)

## 7. 名前と名簿の照合

名前が**テキストレイヤーとして残っている** SVG が必要です。
画像に焼き込んだ / アウトライン化したデータからは文字を読めません。
名簿は Excel（.xlsx）か CSV をアップロードしてください。

In [ ]:
ROSTER_COLUMN = "名前"   # 名簿の列名（列番号なら 0, 1, ... の整数）
STRIP_SPACES  = False    # 「山田 太郎」と「山田太郎」を同一視するなら True

names, warnings = namecheck.extract_names(SVG_FILE, to_mm=doc.to_mm, strip_spaces=STRIP_SPACES)
print(f"SVG 内のテキスト: {len(names)} 件")
for w in warnings:
    print("⚠️", w)

if names:
    print("\n名簿ファイルをアップロードしてください（.xlsx / .csv）")
    up = files.upload()
    roster_file = "/content/" + list(up.keys())[0]
    with open(roster_file, "wb") as f:
        f.write(up[list(up.keys())[0]])

    roster = namecheck.load_roster(roster_file, ROSTER_COLUMN, strip_spaces=STRIP_SPACES)
    print(f"名簿: {len(roster)} 件")

    result = namecheck.compare(names, roster)
    if result.missing:
        print(f"\n❌ 名簿にあるのに SVG にない: {len(result.missing)} 件")
        for x in result.missing:
            print("   ", x)
    if result.extra:
        print(f"\n❌ SVG にあるのに名簿にない: {len(result.extra)} 件")
        for x in result.extra:
            print("   ", x)
    if result.duplicated_in_svg:
        print("\n⚠️ SVG 内で重複:")
        for name, count in result.duplicated_in_svg:
            print(f"    {name} × {count}")
    if result.ok:
        print("\n✅ SVG の名前と名簿は完全に一致しています。")